# Experiment 05 — Client Update Geometry under Statistical Heterogeneity

This notebook studies **how** FedProx changes federated optimization under severe label skew. It complements Experiment 04, which compared predictive performance across a broader proximal-coefficient sweep.

The analysis tracks validation performance, per-client performance, update magnitude, client-to-client alignment, and alignment with the aggregated server update. Update geometry is treated as a **diagnostic**, not as proof of a causal mechanism.

## 1. Research questions and hypotheses

Under a fixed Dirichlet partition with $\alpha=0.1$:

1. Does a larger proximal coefficient $\mu$ reduce client-update magnitude?
2. Does it change agreement between client updates or with the server update?
3. Do those geometric changes coincide with better validation performance or client fairness?

Pre-specified expectations:

- **H1:** stronger proximal regularization should shrink local updates.
- **H2:** alignment may improve when client drift is reduced, but the effect need not be monotonic.
- **H3:** predictive performance and geometric alignment may move differently; correlation will not be interpreted as causation.

The diagnostic comparison uses $\mu\in\{0,0.1,1.0\}$. Here, $\mu=0$ is the FedAvg-equivalent control, $0.1$ is a moderate condition, and $1.0$ is a strong proximal condition.

## 2. Experimental controls

All $\mu$ trajectories reuse the same:

- MNIST preprocessing and `SimpleMLP` architecture;
- saved $\alpha=0.1$ client partition;
- saved initial checkpoint $w_0$;
- optimizer, learning rate, batch size, local epochs, and communication rounds;
- full client participation;
- per-client batch ordering, controlled by seeded `torch.Generator` objects.

The saved client partition is split once into client-local train/validation/test subsets. Training uses only client-train subsets. Model development and comparisons use client-validation subsets. Client-test subsets and the official MNIST test set remain untouched in this notebook.

The seed-42 experiment is a controlled pilot. Multi-seed evidence is required before making general claims.

## 3. Environment and imports

In [ ]:
from pathlib import Path
import copy
import json
import os
import sys
from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms


def find_repo_root():
    # Locate the repository in local, Kaggle, or Colab environments.
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates += [
        Path("/kaggle/working/federated-learning-under-heterogeneity"),
        Path("/content/federated-learning-under-heterogeneity"),
    ]

    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate.resolve()

    raise RuntimeError(
        "Repository root not found. Open, clone, or upload the complete "
        "federated-learning-under-heterogeneity repository before running this notebook."
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository:", REPO_ROOT)
print("Python    :", sys.executable)

In [ ]:
from src.data import client_label_counts, create_client_loaders, split_client_indices
from src.diagnostics import model_to_vector
from src.experiment import RunResult, run_experiment, set_seed
from src.models import SimpleMLP

print("Project imports successful.")

### Implementation map

The notebook defines the experimental protocol and analysis. Reusable implementation remains under `src/`:

- `src/data.py` — partitions, client splits, and reproducible DataLoaders;
- `src/fedprox.py` — the FedProx local objective and client update;
- `src/training.py` — evaluation and weighted server aggregation;
- `src/diagnostics.py` — parameter-vector and update-geometry utilities;
- `src/experiment.py` — the canonical round and experiment runners.

## 4. Configuration

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42
    num_clients: int = 5
    alpha: float = 0.1

    # Experiment 04 used the broader performance sweep. Experiment 05 uses
    # three diagnostic anchors to control the cost of geometry collection.
    performance_mu_values: tuple = (0.0, 0.001, 0.01, 0.1, 1.0)
    diagnostic_mu_values: tuple = (0.0, 0.1, 1.0)

    local_epochs: int = 5
    num_rounds: int = 20
    batch_size: int = 64
    learning_rate: float = 0.01

    datasets_dir: str = "datasets"
    iid_results_dir: str = "results/iid_baseline"
    dirichlet_results_dir: str = "results/dirichlet_noniid"
    results_dir: str = "results/client_update_geometry_v2"


CFG = ExperimentConfig()


def value_tag(value: float) -> str:
    return f"{value:g}".replace("-", "m").replace(".", "p")


RESULT_DIR = REPO_ROOT / CFG.results_dir
RUN_DIR = (
    RESULT_DIR
    / f"alpha_{value_tag(CFG.alpha)}"
    / f"seed_{CFG.seed}"
)
FIGURE_DIR = RUN_DIR / "figures"

RUN_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

with (RUN_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(asdict(CFG), file, indent=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(CFG)
print("Device :", device)
print("Outputs:", RUN_DIR)

## 5. Dataset and fixed client partition

In [ ]:
def load_mnist(root):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_dataset = datasets.MNIST(
        root=root,
        train=True,
        download=True,
        transform=transform,
    )
    official_test_dataset = datasets.MNIST(
        root=root,
        train=False,
        download=True,
        transform=transform,
    )
    return train_dataset, official_test_dataset


def load_torch_object(path, map_location="cpu"):
    # Load saved artifacts across PyTorch versions.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def verify_partition(client_indices, dataset_size: int):
    flat_indices = [
        int(index)
        for indices in client_indices.values()
        for index in indices
    ]

    checks = {
        "dataset_size": dataset_size,
        "assigned": len(flat_indices),
        "unique": len(set(flat_indices)),
        "all_clients_nonempty": all(len(indices) > 0 for indices in client_indices.values()),
    }

    assert checks["assigned"] == dataset_size
    assert checks["unique"] == dataset_size
    assert checks["all_clients_nonempty"]
    assert min(flat_indices) >= 0
    assert max(flat_indices) < dataset_size
    return checks


set_seed(CFG.seed)
train_ds, official_test_ds = load_mnist(REPO_ROOT / CFG.datasets_dir)

print("Federated source examples:", len(train_ds))
print("Official test examples   :", len(official_test_ds), "(held out)")

In [ ]:
DIRICHLET_ALPHA_DIR = (
    REPO_ROOT
    / CFG.dirichlet_results_dir
    / f"alpha_{value_tag(CFG.alpha)}"
)
CLIENT_INDICES_PATH = DIRICHLET_ALPHA_DIR / "client_indices.pt"

if not CLIENT_INDICES_PATH.exists():
    raise FileNotFoundError(
        f"Missing fixed Experiment 03 partition:\n{CLIENT_INDICES_PATH}"
    )

client_indices = load_torch_object(CLIENT_INDICES_PATH)
partition_checks = verify_partition(client_indices, len(train_ds))

assert len(client_indices) == CFG.num_clients
print(partition_checks)

client_size_df = pd.DataFrame([
    {"client_id": client_id, "num_examples": len(indices)}
    for client_id, indices in sorted(client_indices.items())
])
display(client_size_df)

In [ ]:
label_distribution_df = client_label_counts(train_ds, client_indices)
label_distribution_df.to_csv(
    RUN_DIR / "client_label_distribution.csv",
    index=False,
)
display(label_distribution_df)

## 6. Controlled initialization

In [ ]:
INITIAL_MODEL_PATH = REPO_ROOT / CFG.iid_results_dir / "initial_model.pt"

if not INITIAL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing shared initial checkpoint:\n{INITIAL_MODEL_PATH}"
    )

initial_state = load_torch_object(INITIAL_MODEL_PATH)
loss_fn = nn.CrossEntropyLoss()


def fresh_initial_model():
    model = SimpleMLP(784, 10)
    model.load_state_dict(copy.deepcopy(initial_state))
    return model.to(device)


initial_model_a = fresh_initial_model()
initial_model_b = fresh_initial_model()

assert initial_model_a is not initial_model_b
assert torch.equal(
    model_to_vector(initial_model_a),
    model_to_vector(initial_model_b),
)

print("Shared initialization verified.")

## 7. Client-local train, validation, and test splits

The fixed client assignment is split once using a client-specific seeded generator. This preserves each client's heterogeneous distribution while separating model development from final evaluation.

In [ ]:
(
    client_train_indices,
    client_validation_indices,
    client_test_indices,
) = split_client_indices(
    client_indices=client_indices,
    train_fraction=0.8,
    validation_fraction=0.1,
    seed=CFG.seed,
)

assert set(client_train_indices) == set(client_validation_indices) == set(client_test_indices)

client_validation_loaders = create_client_loaders(
    train_ds=train_ds,
    client_indices=client_validation_indices,
    batch_size=CFG.batch_size,
    seed=CFG.seed,
    shuffle=False,
)
client_test_loaders = create_client_loaders(
    train_ds=train_ds,
    client_indices=client_test_indices,
    batch_size=CFG.batch_size,
    seed=CFG.seed,
    shuffle=False,
)

split_size_rows = []
for client_id in sorted(client_indices):
    split_size_rows.append({
        "client_id": client_id,
        "train": len(client_train_indices[client_id]),
        "validation": len(client_validation_indices[client_id]),
        "local_test": len(client_test_indices[client_id]),
    })

split_size_df = pd.DataFrame(split_size_rows)
assert split_size_df[["train", "validation", "local_test"]].to_numpy().sum() == len(train_ds)
display(split_size_df)

## 8. Smoke test

The smoke test uses 128 training and validation examples per client, one local epoch, one communication round, and two $\mu$ values. It validates orchestration, row counts, round-zero anchoring, and reproducibility. Its metrics are not research results.

In [ ]:
SMOKE_CFG = replace(
    CFG,
    local_epochs=1,
    num_rounds=1,
)
SMOKE_MU_VALUES = (0.0, 0.1)

smoke_train_indices = {
    client_id: list(indices)[:128]
    for client_id, indices in client_train_indices.items()
}
smoke_validation_indices = {
    client_id: list(indices)[:128]
    for client_id, indices in client_validation_indices.items()
}
smoke_validation_loaders = create_client_loaders(
    train_ds=train_ds,
    client_indices=smoke_validation_indices,
    batch_size=SMOKE_CFG.batch_size,
    seed=SMOKE_CFG.seed,
    shuffle=False,
)

smoke_result = run_experiment(
    config=SMOKE_CFG,
    mu_values=SMOKE_MU_VALUES,
    initial_model=fresh_initial_model(),
    train_ds=train_ds,
    client_train_indices=smoke_train_indices,
    validation_loaders=smoke_validation_loaders,
    loss_fn=loss_fn,
    device=device,
    verbose=True,
)

In [ ]:
num_mu = len(SMOKE_MU_VALUES)
num_rounds = SMOKE_CFG.num_rounds
num_clients = len(smoke_validation_loaders)
num_pairs = num_clients * (num_clients - 1) // 2

assert len(smoke_result.performance_history) == num_mu * (num_rounds + 1)
assert len(smoke_result.client_performance_history) == num_mu * (num_rounds + 1) * num_clients
assert len(smoke_result.client_update_diagnostics) == num_mu * num_rounds * num_clients
assert len(smoke_result.pairwise_update_diagnostics) == num_mu * num_rounds * num_pairs
assert len(smoke_result.round_diagnostics) == num_mu * num_rounds

round_zero = smoke_result.performance_history.query("round == 0").copy()
anchor_metrics = [
    "weighted_loss",
    "weighted_accuracy",
    "mean_client_accuracy",
    "worst_client_accuracy",
]
assert (round_zero[anchor_metrics].nunique(dropna=False) == 1).all()

print("Smoke-test row counts and round-zero anchors passed.")
display(round_zero[["mu", *anchor_metrics]])

In [ ]:
repeat_smoke_result = run_experiment(
    config=SMOKE_CFG,
    mu_values=SMOKE_MU_VALUES,
    initial_model=fresh_initial_model(),
    train_ds=train_ds,
    client_train_indices=smoke_train_indices,
    validation_loaders=smoke_validation_loaders,
    loss_fn=loss_fn,
    device=device,
    verbose=False,
)

for field_name in RunResult.__dataclass_fields__:
    pd.testing.assert_frame_equal(
        getattr(smoke_result, field_name),
        getattr(repeat_smoke_result, field_name),
    )

print("Full smoke-run reproducibility check passed.")

## 9. Full diagnostic experiment

Run this section on a GPU-backed environment. Set `RUN_FULL_EXPERIMENT=True` only after the smoke test passes. The canonical runner evaluates validation data at round 0 and after each aggregated round; it never accesses either test split.

The five raw tables are saved immediately after the run.

In [ ]:
RUN_FULL_EXPERIMENT = False  # Set to True on Kaggle after the smoke test passes.

RESULT_PATHS = {
    "performance_history": RUN_DIR / "performance_history.csv",
    "client_performance_history": RUN_DIR / "client_performance_history.csv",
    "client_update_diagnostics": RUN_DIR / "client_update_diagnostics.csv",
    "pairwise_update_diagnostics": RUN_DIR / "pairwise_update_diagnostics.csv",
    "round_diagnostics": RUN_DIR / "round_diagnostics.csv",
}

full_result = None

if RUN_FULL_EXPERIMENT:
    full_result = run_experiment(
        config=CFG,
        mu_values=CFG.diagnostic_mu_values,
        initial_model=fresh_initial_model(),
        train_ds=train_ds,
        client_train_indices=client_train_indices,
        validation_loaders=client_validation_loaders,
        loss_fn=loss_fn,
        device=device,
        verbose=True,
    )

    for field_name, output_path in RESULT_PATHS.items():
        getattr(full_result, field_name).to_csv(output_path, index=False)

    print("Saved full experiment to:", RUN_DIR)
elif all(path.exists() for path in RESULT_PATHS.values()):
    full_result = RunResult(**{
        field_name: pd.read_csv(path)
        for field_name, path in RESULT_PATHS.items()
    })
    print("Loaded existing experiment artifacts from:", RUN_DIR)
else:
    print("Full experiment not run. Set RUN_FULL_EXPERIMENT=True on Kaggle.")

## 10. Artifact integrity

In [ ]:
if full_result is not None:
    num_mu = len(CFG.diagnostic_mu_values)
    num_rounds = CFG.num_rounds
    num_clients = len(client_validation_loaders)
    num_pairs = num_clients * (num_clients - 1) // 2

    expected_rows = {
        "performance_history": num_mu * (num_rounds + 1),
        "client_performance_history": num_mu * (num_rounds + 1) * num_clients,
        "client_update_diagnostics": num_mu * num_rounds * num_clients,
        "pairwise_update_diagnostics": num_mu * num_rounds * num_pairs,
        "round_diagnostics": num_mu * num_rounds,
    }

    integrity_rows = []
    for field_name, expected in expected_rows.items():
        table = getattr(full_result, field_name)
        integrity_rows.append({
            "artifact": field_name,
            "rows": len(table),
            "expected_rows": expected,
            "passed": len(table) == expected,
        })

    integrity_df = pd.DataFrame(integrity_rows)
    assert integrity_df["passed"].all()
    display(integrity_df)
else:
    print("No full-run artifacts available for validation.")

## 11. Validation performance and client fairness

Weighted accuracy measures performance across examples, so larger clients contribute more. Worst-client accuracy exposes whether strong aggregate performance hides a poorly served client.

In [ ]:
if full_result is not None:
    performance_df = full_result.performance_history.copy()
    final_performance_df = (
        performance_df.loc[performance_df["round"] == CFG.num_rounds]
        .sort_values("mu")
        .reset_index(drop=True)
    )

    display(final_performance_df[[
        "mu",
        "weighted_loss",
        "weighted_accuracy",
        "mean_client_accuracy",
        "std_client_accuracy",
        "worst_client_accuracy",
        "p10_client_accuracy",
    ]])
else:
    print("Run or load the full experiment first.")

In [ ]:
if full_result is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

    for mu_value in sorted(performance_df["mu"].unique()):
        trajectory = performance_df.loc[performance_df["mu"] == mu_value]
        axes[0].plot(
            trajectory["round"],
            trajectory["weighted_accuracy"],
            label=f"mu={mu_value:g}",
        )
        axes[1].plot(
            trajectory["round"],
            trajectory["worst_client_accuracy"],
            label=f"mu={mu_value:g}",
        )

    axes[0].set_title("Weighted validation accuracy")
    axes[1].set_title("Worst-client validation accuracy")

    for axis in axes:
        axis.set_xlabel("Communication round")
        axis.set_ylabel("Accuracy")
        axis.grid(alpha=0.25)

    axes[1].legend()
    fig.suptitle(f"Validation performance under label skew (alpha={CFG.alpha:g})")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "validation_performance_vs_round.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("Run or load the full experiment first.")

## 12. Client-update geometry

For client $k$ at round $t$, the update is $\Delta_k=w_{k,t+1}-w_t$.

- **Update magnitude:** $\lVert\Delta_k\rVert_2$.
- **Relative update magnitude:** $\lVert\Delta_k\rVert_2/(\lVert w_t\rVert_2+\epsilon)$.
- **Aggregate alignment:** cosine similarity between $\Delta_k$ and the weighted server update.
- **Leave-one-out alignment:** cosine similarity between $\Delta_k$ and the weighted aggregate of the other clients only. This removes the client's automatic self-contribution to the server update.
- **Pairwise similarity:** cosine similarity between two client updates.

These quantities describe optimization behavior; they do not directly measure client utility or fairness.

In [ ]:
if full_result is not None:
    client_geometry_df = full_result.client_update_diagnostics.copy()
    pairwise_geometry_df = full_result.pairwise_update_diagnostics.copy()

    geometry_by_round = (
        client_geometry_df
        .groupby(["mu", "round"], as_index=False)
        .agg(
            median_update_magnitude=("update_magnitude", "median"),
            mean_loo_alignment=("loo_alignment", "mean"),
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

    for mu_value in sorted(geometry_by_round["mu"].unique()):
        trajectory = geometry_by_round.loc[geometry_by_round["mu"] == mu_value]
        axes[0].plot(
            trajectory["round"],
            trajectory["median_update_magnitude"],
            label=f"mu={mu_value:g}",
        )
        axes[1].plot(
            trajectory["round"],
            trajectory["mean_loo_alignment"],
            label=f"mu={mu_value:g}",
        )

    axes[0].set_title("Median client-update magnitude")
    axes[1].set_title("Mean leave-one-out alignment")

    for axis in axes:
        axis.set_xlabel("Communication round")
        axis.grid(alpha=0.25)

    axes[0].set_ylabel("L2 norm")
    axes[1].set_ylabel("Cosine similarity")
    axes[1].axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
    axes[1].legend()

    fig.suptitle(f"Client-update geometry (alpha={CFG.alpha:g}, E={CFG.local_epochs})")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "client_update_geometry_vs_round.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("Run or load the full experiment first.")

## 13. Compact comparison table

In [ ]:
if full_result is not None:
    final_client_geometry = (
        client_geometry_df.loc[client_geometry_df["round"] == CFG.num_rounds]
        .groupby("mu", as_index=False)
        .agg(
            median_update_magnitude=("update_magnitude", "median"),
            mean_loo_alignment=("loo_alignment", "mean"),
        )
    )
    final_pairwise_geometry = (
        pairwise_geometry_df.loc[pairwise_geometry_df["round"] == CFG.num_rounds]
        .groupby("mu", as_index=False)
        .agg(mean_pairwise_similarity=("cosine_similarity", "mean"))
    )

    comparison_df = (
        final_performance_df[[
            "mu",
            "weighted_accuracy",
            "worst_client_accuracy",
        ]]
        .merge(final_client_geometry, on="mu", how="inner")
        .merge(final_pairwise_geometry, on="mu", how="inner")
        .sort_values("mu")
        .reset_index(drop=True)
    )
    comparison_df.to_csv(RUN_DIR / "comparison_summary.csv", index=False)
    display(comparison_df)
else:
    print("Run or load the full experiment first.")

## 14. Interpretation guide

Evaluate the hypotheses using the complete trajectories, not one visually convenient round:

1. **Magnitude:** Does increasing $\mu$ consistently shrink median or per-client update magnitudes?
2. **Agreement:** Do leave-one-out and pairwise similarities change consistently, or only for selected clients and rounds?
3. **Performance:** Does a geometric change coincide with higher weighted validation accuracy?
4. **Fairness:** Does it also improve worst-client and low-percentile accuracy?
5. **Alternative explanations:** Could apparent improvement arise from sample-size weighting, class rebalancing, or trajectory divergence rather than geometry itself?

Use language such as “is associated with” or “coincides with.” This experiment does not identify a causal effect of alignment on accuracy.

## 15. Limitations and next steps

- The current experiment uses one seed, one saved partition, five clients, MNIST, full participation, and one architecture.
- Client rounds are not independent experimental units; the seed/partition combination is the experimental unit.
- Geometry metrics compress a high-dimensional update into scalar summaries and may omit important structure.
- Validation results guide model comparison; neither local-test nor official-test results are reported here.
- Statistical heterogeneity is isolated, while communication failures, stragglers, privacy, and malicious behavior are out of scope.

Next steps:

1. Repeat the controlled comparison across multiple seeds and partitions.
2. Run a matched-anchor counterfactual study that compares $\mu$ values from identical round-start checkpoints and batch orders.
3. Study SCAFFOLD as a method designed specifically to correct client drift.
4. Add partial participation and communication-cost measurements only after the current pipeline remains stable.

## 16. Saved artifacts

In [ ]:
artifact_rows = [
    {"artifact": "configuration", "path": RUN_DIR / "config.json"},
    {"artifact": "fixed partition", "path": CLIENT_INDICES_PATH},
    {"artifact": "label distribution", "path": RUN_DIR / "client_label_distribution.csv"},
    *[
        {"artifact": field_name, "path": path}
        for field_name, path in RESULT_PATHS.items()
    ],
    {"artifact": "comparison summary", "path": RUN_DIR / "comparison_summary.csv"},
    {"artifact": "figures", "path": FIGURE_DIR},
]

artifact_df = pd.DataFrame(artifact_rows)
artifact_df["exists"] = artifact_df["path"].map(lambda path: Path(path).exists())
artifact_df["path"] = artifact_df["path"].map(str)
display(artifact_df)